# LEER ALUMNOS

### LIBRERIAS

In [1]:
import pandas as pd

In [7]:
pip install --upgrade pip


     ---------------------------------------- 1.8/1.8 MB 6.7 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 22.3
    Uninstalling pip-22.3:
      Successfully uninstalled pip-22.3
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [41]:
import sys
!{sys.executable} -m pip install openpyxl

!{sys.executable} -m pip install deltalake
!{sys.executable} -m pip install pyarrow

   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   - -------------------------------------- 1.3/27.5 MB 7.4 MB/s eta 0:00:04
   ---- ----------------------------------- 3.4/27.5 MB 9.1 MB/s eta 0:00:03
   -------- ------------------------------- 5.8/27.5 MB 10.1 MB/s eta 0:00:03
   ------------ --------------------------- 8.4/27.5 MB 10.6 MB/s eta 0:00:02
   --------------- ------------------------ 10.5/27.5 MB 10.7 MB/s eta 0:00:02
   ------------------ --------------------- 12.8/27.5 MB 10.5 MB/s eta 0:00:02
   ---------------------- ----------------- 15.2/27.5 MB 10.5 MB/s eta 0:00:02
   ------------------------- -------------- 17.3/27.5 MB 10.5 MB/s eta 0:00:01
   ---------------------------- ----------- 19.7/27.5 MB 10.6 MB/s eta 0:00:01
   ------------------------------- -------- 21.8/27.5 MB 10.5 MB/s eta 0:00:01
   ---------------------------------- ----- 23.9/27.5 MB 10.4 MB/s eta 0:00:01
   -------------------------------------- - 26.2/27.5 MB 10.5 MB/s 

In [42]:
# Librerias necesarias para el almacenamiento
from deltalake import write_deltalake, DeltaTable
import pyarrow as pa

### FUNCIONES

In [36]:
# Funciones para la ingesta de datos
def get_full_data(base_url, endpoint, data_field=None, params=None, headers=None):
    """
    Realiza una solicitud GET a una API para obtener datos.

    Parámetros:
    base_url (str): La URL base de la API.
    endpoint (str): El endpoint de la API al que se realizará la solicitud.
    params (dict): Parámetros de consulta para enviar con la solicitud.
    data_field (str): El nombre del campo en el JSON que contiene los datos.
    headers (dict): Encabezados para enviar con la solicitud.

    Retorna:
    dict: Los datos obtenidos de la API en formato JSON.
    """
    if params==None:
        params={}

    endpoint_url = f"{base_url}/{endpoint}"
    # Arrays usdados para almacenar los resultados de todas las paginas que devuelva la API
    results=[]
    all_results=[]
    pagina=1
    try:
        while True:
            # En cada iteracion se recorre una pagina con 1000 registros, se los almacena y se pasa a la siguiente pagina.
            params_page={
                "page": pagina,
                "limit": 1000,
                "sort": "period.start", 
                "order": "asc"  
            }
            params = params | params_page
            response = requests.get(endpoint_url, params=params, headers=headers)
            response.raise_for_status()  
            try:
                data = response.json()
                if data_field:
                    results= data.get(data_field,[])
                    all_results.extend(results)
            except:
                print("El formato de respuesta no es el esperado")
                return None
            if len(results) < params["limit"]:
                break
            pagina+=1
    
        return all_results
        # Verificar si los datos están en formato JSON.

    except requests.exceptions.RequestException as e:
        # Capturar cualquier error de solicitud, como errores HTTP.
        print(f"La petición ha fallado. Código de error : {e}")
        return None

def build_table(json_data):
    """
    Construye un DataFrame de pandas a partir de datos en formato JSON.

    Parámetros:
    json_data (dict): Los datos en formato JSON obtenidos de una API.

    Retorna:
    DataFrame: Un DataFrame de pandas que contiene los datos.
    """
    try:
        df = pd.json_normalize(json_data, sep='.', record_path=None)
        return df
    except:
        print("Los datos no están en el formato esperado")
        return None
    

In [37]:
# Funciones para el almacenamiento de datos
def save_data_as_delta(df, path, mode="overwrite", partition_cols=None):
    """
    Guarda un dataframe en formato Delta Lake en la ruta especificada.
    A su vez, es capaz de particionar el dataframe por una o varias columnas.
    Por defecto, el modo de guardado es "overwrite".

    Args:
      df (pd.DataFrame): El dataframe a guardar.
      path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      mode (str): El modo de guardado. Son los modos que soporta la libreria
      deltalake: "overwrite", "append", "error", "ignore".
      partition_cols (list or str): La/s columna/s por las que se particionará el
      dataframe. Si no se especifica, no se particionará.
    """
    try:
       write_deltalake(
        path, df, mode=mode, partition_by=partition_cols
        )
       print("Datos almacenados en el Datalke en:", path)
    except Exception as e:
        print("Error al cargar datos en Deltake: ", e)

def save_new_data_as_delta(new_data, data_path, predicate, partition_cols=None):
    """
    Guarda solo nuevos datos en formato Delta Lake usando la operación MERGE,
    comparando los datos ya cargados con los datos que se desean almacenar
    asegurando que no se guarden registros duplicados.

    Args:
      new_data (pd.DataFrame): Los datos que se desean guardar.
      data_path (str): La ruta donde se guardará el dataframe en formato Delta Lake.
      predicate (str): La condición de predicado para la operación MERGE.
    """

    try:
      dt = DeltaTable(data_path)
      new_data_pa = pa.Table.from_pandas(new_data)
      # Se insertan en target, datos de source que no existen en target
      dt.merge(
          source=new_data_pa,
          source_alias="source",
          target_alias="target",
          predicate=predicate
      ) \
      .when_not_matched_insert_all() \
      .execute()
    # Si no existe la tabla Delta Lake, se guarda como nueva
    except:
      save_data_as_delta(new_data, data_path, partition_cols=partition_cols)

## BRONZE

### INGESTA

In [16]:
# 1. Leer el Excel
df_students = pd.read_excel("Dataset\Alumnos_Libres_sql.xlsx")

df_students


,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2
3,2025,5,Ingeniería en Sistemas de Información ...,2008,111,Inglés I ...,27,8,5,13,3,0,19,4,2
4,2025,5,Ingeniería en Sistemas de Información ...,2008,115,Sistemas de Representación ...,43,8,8,33,0,0,35,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


### ALMACENAMIENTO

In [21]:
print(df_students.dtypes)

Año                                 int64
especialid                          int64
esp                                object
plan                                int64
materia                             int64
nombre                             object
Total Inscriptos                    int64
Nuevos Inscriptos                   int64
Nuevos Inscriptos Libres            int64
Recursantes Libres                  int64
Nuevos Inscriptos Regulares         int64
Nuevos Inscritpos Promocionados     int64
Recursantes                         int64
Recursantes Regulares               int64
Recursantes Promocionados           int64
dtype: object


In [29]:
# Nos aseguramos de que ninguna columna tenga nulos y definimos los tipos correctos para almacenarlos
df_students["Año"] = df_students["Año"].fillna(0).astype(int)
df_students["especialid"] = df_students["especialid"].fillna(0).astype(int)
df_students["esp"] = df_students["esp"].fillna("").astype(str)
df_students["plan"] = df_students["plan"].fillna(0).astype(int)
df_students["materia"] = df_students["materia"].fillna(0).astype(int)
df_students["nombre"] = df_students["nombre"].fillna("").astype(str)
df_students["Total Inscriptos"] = df_students["Total Inscriptos"].fillna(0).astype(int)
df_students["Nuevos Inscriptos"] = df_students["Nuevos Inscriptos"].fillna(0).astype(int)
df_students["Nuevos Inscriptos Libres"] = df_students["Nuevos Inscriptos Libres"].fillna(0).astype(int)
df_students["Recursantes Libres"] = df_students["Recursantes Libres"].fillna(0).astype(int)
df_students["Nuevos Inscriptos Regulares"] = df_students["Nuevos Inscriptos Regulares"].fillna(0).astype(int)
df_students["Nuevos Inscritpos Promocionados"] = df_students["Nuevos Inscritpos Promocionados"].fillna(0).astype(int)
df_students["Recursantes"] = df_students["Recursantes"].fillna(0).astype(int)
df_students["Recursantes Regulares"] = df_students["Recursantes Regulares"].fillna(0).astype(int)
df_students["Recursantes Promocionados"] = df_students["Recursantes Promocionados"].fillna(0).astype(int)
df_students

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2
3,2025,5,Ingeniería en Sistemas de Información ...,2008,111,Inglés I ...,27,8,5,13,3,0,19,4,2
4,2025,5,Ingeniería en Sistemas de Información ...,2008,115,Sistemas de Representación ...,43,8,8,33,0,0,35,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


In [34]:
# Filtramos los alumnos del plan 2008
df_plan_2008 = df_students[df_students["plan"] == 2008].copy()

# Filtramos los alumnos del plan 2023
df_plan_2023 = df_students[df_students["plan"] == 2023].copy()

# Vemos uno de los plnes
df_plan_2023

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados
60,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12
61,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9
62,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14
63,2025,5,Ingeniería en Sistemas de Información ...,2023,104,Inglés I ...,467,424,143,18,79,202,43,10,15
64,2025,5,Ingeniería en Sistemas de Información ...,2023,105,Lógica y Estructuras Discretas ...,982,533,288,285,180,65,449,126,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0
364,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0
365,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0
366,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0


In [43]:
# Almacenamos los datos en un Deltalake
path_2008="Data/bronze/2008"
path_2023="Data/bronze/2023"

save_data_as_delta(df_plan_2008, path_2008, mode="overwrite")
save_data_as_delta(df_plan_2023, path_2023, mode="overwrite")

Datos almacenados en el Datalke en: Data/bronze/2008
Datos almacenados en el Datalke en: Data/bronze/2023


In [54]:
# Visualizacion de todos los datos almacenados
dsc_data_2008="Data/bronze/2008"
df_students_2008_stored = DeltaTable(dsc_data_2008).to_pandas() 
df_students_2008_stored[:3]

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados,__index_level_0__
0,2025,5,Ingeniería en Sistemas de Información ...,2008,101,Álgebra y Geometría Analítica ...,87,5,5,72,0,0,82,8,2,0
1,2025,5,Ingeniería en Sistemas de Información ...,2008,102,Análisis Matemático I ...,137,13,11,120,1,1,124,3,1,1
2,2025,5,Ingeniería en Sistemas de Información ...,2008,110,Ingeniería y Sociedad ...,22,2,2,17,0,0,20,1,2,2


## Silver

In [56]:
dsc_data_bronze_2008="Data/bronze/2008"
dsc_data_bronze_2023="Data/bronze/2023"
df_students_2008_bronze = DeltaTable(dsc_data_bronze_2008).to_pandas() 
df_students_2023_bronze = DeltaTable(dsc_data_bronze_2023).to_pandas() 

df_students_2023_bronze.head(3)

,Año,especialid,esp,plan,materia,nombre,Total Inscriptos,Nuevos Inscriptos,Nuevos Inscriptos Libres,Recursantes Libres,Nuevos Inscriptos Regulares,Nuevos Inscritpos Promocionados,Recursantes,Recursantes Regulares,Recursantes Promocionados,__index_level_0__
0,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12,60
1,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9,61
2,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14,62


In [57]:
df_students_2008_bronze.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 16 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Año                              60 non-null     int64 
 1   especialid                       60 non-null     int64 
 2   esp                              60 non-null     object
 3   plan                             60 non-null     int64 
 4   materia                          60 non-null     int64 
 5   nombre                           60 non-null     object
 6   Total Inscriptos                 60 non-null     int64 
 7   Nuevos Inscriptos                60 non-null     int64 
 8   Nuevos Inscriptos Libres         60 non-null     int64 
 9   Recursantes Libres               60 non-null     int64 
 10  Nuevos Inscriptos Regulares      60 non-null     int64 
 11  Nuevos Inscritpos Promocionados  60 non-null     int64 
 12  Recursantes                      60 no

In [68]:
df_students_2008_cleaned = df_students_2008_bronze
df_students_2023_cleaned  = df_students_2023_bronze


# Cambiar el nombres
# Diccionario maestro para estandarizar y corregir los nombres de las columnas
renombres_columnas = {
    # 1. Quitar caracteres especiales (como la 'ñ') y hacer los nombres más explícitos
    "Año": "Anio",
    "especialid": "Especialidad_ID",
    "esp": "Especialidad_Nombre",
    "plan": "Plan",
    "materia": "Materia_ID",
    "nombre": "Materia_Nombre",
    
    # 2. Reemplazar espacios por guiones bajos (_)
    "Total Inscriptos": "Total_Inscriptos",
    "Nuevos Inscriptos": "Nuevos_Inscriptos",
    "Nuevos Inscriptos Libres": "Nuevos_Inscriptos_Libres",
    "Recursantes Libres": "Recursantes_Libres",
    "Nuevos Inscriptos Regulares": "Nuevos_Inscriptos_Regulares",
    
    # 3. Corregir el error tipográfico ("Inscritpos" -> "Inscriptos") 
    "Nuevos Inscritpos Promocionados": "Nuevos_Inscriptos_Promocionados",
    
    # 4. Asegurarnos de que tengan la primera letra en mayúscula por convención
    "Recursantes": "Recursantes",
    "Recursantes Regulares": "Recursantes_Regulares",
    "Recursantes Promocionados": "Recursantes_Promocionados"
}

# Para el plan 2008
df_students_2008_cleaned = df_students_2008_cleaned.rename(columns=renombres_columnas)

# Para el plan 2023
df_students_2023_cleaned = df_students_2023_cleaned.rename(columns=renombres_columnas)
df_students_2023_cleaned


,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,__index_level_0__
0,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12,60
1,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9,61
2,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14,62
3,2025,5,Ingeniería en Sistemas de Información ...,2023,104,Inglés I ...,467,424,143,18,79,202,43,10,15,63
4,2025,5,Ingeniería en Sistemas de Información ...,2023,105,Lógica y Estructuras Discretas ...,982,533,288,285,180,65,449,126,38,64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0,363
122,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0,364
123,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0,365
124,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0,366


In [ ]:
# -----------------------------
# Para el plan 2008
# -----------------------------
# Verificamos si la columna 'ID_Compuesto' NO existe adentro del DataFrame
if "ID_Compuesto" not in df_students_2008_cleaned.columns:
    
    # 1. Creamos directamente la columna nueva
    df_students_2008_cleaned["ID_Compuesto"] = (
        df_students_2008_cleaned["Especialidad_ID"].astype(str) + "_" +
        df_students_2008_cleaned["Plan"].astype(str) + "_" +
        df_students_2008_cleaned["Anio"].astype(str)
    )
    
    # 2. Si la columna basura '__index_level_0__' existe, la eliminamos
    if "__index_level_0__" in df_students_2008_cleaned.columns:
        df_students_2008_cleaned = df_students_2008_cleaned.drop(columns=["__index_level_0__"])


# -----------------------------
# Para el plan 2023
# -----------------------------
if "ID_Compuesto" not in df_students_2023_cleaned.columns:
    
    df_students_2023_cleaned["ID_Compuesto"] = (
        df_students_2023_cleaned["Especialidad_ID"].astype(str) + "_" +
        df_students_2023_cleaned["Plan"].astype(str) + "_" +
        df_students_2023_cleaned["Anio"].astype(str)
    )
    
    if "__index_level_0__" in df_students_2023_cleaned.columns:
        df_students_2023_cleaned = df_students_2023_cleaned.drop(columns=["__index_level_0__"])
        
df_students_2023_cleaned

,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,ID_Compuesto
0,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12,5_2023_2025
1,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9,5_2023_2025
2,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14,5_2023_2025
3,2025,5,Ingeniería en Sistemas de Información ...,2023,104,Inglés I ...,467,424,143,18,79,202,43,10,15,5_2023_2025
4,2025,5,Ingeniería en Sistemas de Información ...,2023,105,Lógica y Estructuras Discretas ...,982,533,288,285,180,65,449,126,38,5_2023_2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,2025,31,Ingeniería Civil ...,2023,306,Cálculo Avanzado ...,7,7,0,0,4,3,0,0,0,31_2023_2025
122,2025,31,Ingeniería Civil ...,2023,307,Instalaciones Eléctricas y Acústicas ...,9,9,3,0,3,3,0,0,0,31_2023_2025
123,2025,31,Ingeniería Civil ...,2023,308,Instalaciones Termomecánicas ...,8,8,1,0,7,0,0,0,0,31_2023_2025
124,2025,31,Ingeniería Civil ...,2023,309,Economía ...,42,42,31,0,8,3,0,0,0,31_2023_2025


In [80]:
# -----------------------------
# Para el plan 2008
# -----------------------------
# Total Libres = Nuevos_Inscriptos_Libres + Recursantes_Libres
if "Total_Libres" not in df_students_2008_cleaned.columns:
    df_students_2008_cleaned["Total_Libres"] = (
        df_students_2008_cleaned["Nuevos_Inscriptos_Libres"] + 
        df_students_2008_cleaned["Recursantes_Libres"]
    )

# Total Regulares = Nuevos_Inscriptos_Regulares + Recursantes_Regulares
if "Total_Regulares" not in df_students_2008_cleaned.columns:
    df_students_2008_cleaned["Total_Regulares"] = (
        df_students_2008_cleaned["Nuevos_Inscriptos_Regulares"] + 
        df_students_2008_cleaned["Recursantes_Regulares"]
    )

# Total Promocionados = Nuevos_Inscriptos_Promocionados + Recursantes_Promocionados
if "Total_Promocionados" not in df_students_2008_cleaned.columns:
    df_students_2008_cleaned["Total_Promocionados"] = (
        df_students_2008_cleaned["Nuevos_Inscriptos_Promocionados"] + 
        df_students_2008_cleaned["Recursantes_Promocionados"]
    )


# -----------------------------
# Para el plan 2023
# -----------------------------
# Total Libres
if "Total_Libres" not in df_students_2023_cleaned.columns:
    df_students_2023_cleaned["Total_Libres"] = (
        df_students_2023_cleaned["Nuevos_Inscriptos_Libres"] + 
        df_students_2023_cleaned["Recursantes_Libres"]
    )

# Total Regulares
if "Total_Regulares" not in df_students_2023_cleaned.columns:
    df_students_2023_cleaned["Total_Regulares"] = (
        df_students_2023_cleaned["Nuevos_Inscriptos_Regulares"] + 
        df_students_2023_cleaned["Recursantes_Regulares"]
    )

# Total Promocionados
if "Total_Promocionados" not in df_students_2023_cleaned.columns:
    df_students_2023_cleaned["Total_Promocionados"] = (
        df_students_2023_cleaned["Nuevos_Inscriptos_Promocionados"] + 
        df_students_2023_cleaned["Recursantes_Promocionados"]
    )
df_students_2023_cleaned[df_students_2023_cleaned["Especialidad_ID"]==5]

,Anio,Especialidad_ID,Especialidad_Nombre,Plan,Materia_ID,Materia_Nombre,Total_Inscriptos,Nuevos_Inscriptos,Nuevos_Inscriptos_Libres,Recursantes_Libres,Nuevos_Inscriptos_Regulares,Nuevos_Inscriptos_Promocionados,Recursantes,Recursantes_Regulares,Recursantes_Promocionados,ID_Compuesto,Total_Libres,Total_Regulares,Total_Promocionados
0,2025,5,Ingeniería en Sistemas de Información ...,2023,101,Análisis Matemático I ...,1100,697,553,340,104,40,403,51,12,5_2023_2025,893,155,52
1,2025,5,Ingeniería en Sistemas de Información ...,2023,102,Álgebra y Geometría Analítica ...,927,633,418,228,152,63,294,57,9,5_2023_2025,646,209,72
2,2025,5,Ingeniería en Sistemas de Información ...,2023,103,Física I ...,891,516,335,245,138,43,375,116,14,5_2023_2025,580,254,57
3,2025,5,Ingeniería en Sistemas de Información ...,2023,104,Inglés I ...,467,424,143,18,79,202,43,10,15,5_2023_2025,161,89,217
4,2025,5,Ingeniería en Sistemas de Información ...,2023,105,Lógica y Estructuras Discretas ...,982,533,288,285,180,65,449,126,38,5_2023_2025,573,306,103
5,2025,5,Ingeniería en Sistemas de Información ...,2023,106,Algoritmos y Estructuras de Datos ...,901,527,289,255,62,176,374,46,73,5_2023_2025,544,108,249
6,2025,5,Ingeniería en Sistemas de Información ...,2023,107,Arquitectura de Computadoras ...,965,535,345,315,109,81,430,86,29,5_2023_2025,660,195,110
7,2025,5,Ingeniería en Sistemas de Información ...,2023,108,Sistemas y Procesos de Negocio ...,819,542,231,193,39,272,277,18,66,5_2023_2025,424,57,338
8,2025,5,Ingeniería en Sistemas de Información ...,2023,201,Análisis Matemático II ...,394,371,254,14,114,3,23,9,0,5_2023_2025,268,123,3
9,2025,5,Ingeniería en Sistemas de Información ...,2023,202,Física II ...,386,344,201,25,108,35,42,13,4,5_2023_2025,226,121,39


In [73]:
# Almacenamos los datos en un Deltalake
path_2008="Data/silver/2008"
path_2023="Data/silver/2023"

save_data_as_delta(df_students_2008_cleaned, path_2008, mode="overwrite")
save_data_as_delta(df_students_2023_cleaned, path_2023, mode="overwrite")

Datos almacenados en el Datalke en: Data/silver/2008
Datos almacenados en el Datalke en: Data/silver/2023


In [75]:
df_students_2008_cleaned["Especialidad_Nombre"]

0     Ingeniería en Sistemas de Información         ...
1     Ingeniería en Sistemas de Información         ...
2     Ingeniería en Sistemas de Información         ...
3     Ingeniería en Sistemas de Información         ...
4     Ingeniería en Sistemas de Información         ...
5     Ingeniería en Sistemas de Información         ...
6     Ingeniería en Sistemas de Información         ...
7     Ingeniería en Sistemas de Información         ...
8     Ingeniería en Sistemas de Información         ...
9     Ingeniería en Sistemas de Información         ...
10    Ingeniería en Sistemas de Información         ...
11    Ingeniería en Sistemas de Información         ...
12    Ingeniería en Sistemas de Información         ...
13    Ingeniería en Sistemas de Información         ...
14    Ingeniería en Sistemas de Información         ...
15    Ingeniería en Sistemas de Información         ...
16    Ingeniería en Sistemas de Información         ...
17    Ingeniería en Sistemas de Información     